In [15]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic
import pandas as pd
from dotenv import load_dotenv
import os
from datasets import Dataset

In [16]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [27]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [19]:
# Set up Claude as the evaluator LLM
claude_llm = ChatAnthropic(api_key=anthropic_key, model="claude-3-5-sonnet-20240620")
evaluator_llm = LangchainLLMWrapper(claude_llm)

In [24]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import Faithfulness

sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = Faithfulness(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

1.0

In [41]:
# Function to query the LLM with hallucination-aware prompt
async def evaluate_hallucination(context, question, response):
    
    from ragas.dataset_schema import SingleTurnSample 
    from ragas.metrics import Faithfulness

    sample = SingleTurnSample(
            user_input=question,
            response=response,
            retrieved_contexts=context
        )
    scorer = Faithfulness(llm=evaluator_llm)
    faithfulness_score = await scorer.single_turn_ascore(sample)
    return faithfulness_score

In [42]:
# Initialize list to hold results
results = []

# Iterate through dataset (starting with 10 records)
for i in range(0,10):

    # Define context and question
    context = [qa_data.knowledge[i]]
    question = qa_data.question[i]
    hallucinated_response = qa_data.hallucinated_answer[i]
    
    # Get results
    faithfulness_score = evaluate_hallucination(context, question, hallucinated_response)
    
    # Store results
    results.append({
        "question": question,
        "context": context,
        "llm_answer": hallucinated_response,
        "hallucination_score": faithfulness_score
    })

In [54]:
import asyncio

# Initialize list to hold results
results = []

# Define an async function to run all evaluations
async def evaluate_all():
    tasks = []
    
    # Iterate through dataset (first 10 records)
    for i in range(100):
        context = [qa_data.knowledge[i]]
        question = qa_data.question[i]
        hallucinated_response = qa_data.hallucinated_answer[i]
        
        # Store async tasks
        tasks.append(
            evaluate_hallucination(context, question, hallucinated_response)
        )

    # Run all evaluations concurrently
    faithfulness_scores = await asyncio.gather(*tasks)
    
    # Store results
    for i in range(10):
        results.append({
            "question": qa_data.question[i],
            "context": [qa_data.knowledge[i]],
            "llm_answer": qa_data.hallucinated_answer[i],
            "faithfulness_score": faithfulness_scores[i]
        })

# Run the async function properly
asyncio.run(evaluate_all())

In [55]:
# Convert to df, get hallucination score
results_df = pd.DataFrame(results)
results_df['hallucination_score'] = 1 - results_df.faithfulness_score

In [57]:
results_df

,question,context,llm_answer,faithfulness_score,hallucination_score
0,Which magazine was started first Arthur's Maga...,[Arthur's Magazine (1844–1846) was an American...,First for Women was started first.,0.0,1.0
1,The Oberoi family is part of a hotel company t...,[The Oberoi family is an Indian family that is...,The Oberoi family's hotel company is based in ...,0.5,0.5
2,Musician and satirist Allie Goertz wrote a son...,"[Allison Beth ""Allie"" Goertz (born March 2, 19...","Allie Goertz wrote a song about Milhouse, a po...",0.8,0.2
3,What nationality was James Henry Miller's wife?,"[Margaret ""Peggy"" Seeger (born June 17, 1935) ...",James Henry Miller's wife was British.,0.0,1.0
4,Cadmium Chloride is slightly soluble in this c...,[ It is a hygroscopic solid that is highly sol...,water with a hint of alcohol,0.0,1.0
5,Which tennis player won more Grand Slam titles...,"[Jonathan Stark (born April 3, 1971) is a form...",Henri Leconte won more Grand Slam titles.,0.0,1.0
6,Which genus of moth in the world's seventh-lar...,[Indogrammodes is a genus of moths of the Cram...,The Indogrammodes genus of moths found in Indi...,1.0,0.0
7,Who was once considered the best kick boxer in...,[ Fighters from around world on the roster inc...,Badr Hari is a notorious kickboxer.,0.5,0.5
8,"The Dutch-Belgian television series that ""Hous...",[House of Anubis is a mystery television serie...,"The inspiration for ""House of Anubis"" first ai...",0.5,0.5
9,What is the length of the track where the 2013...,[The 2013 Liqui Moly Bathurst 12 Hour was an e...,The Mount Panorama Circuit track is longer tha...,0.5,0.5
